# Source data discovery and inventory

This notebook inspects both implemented datasets and sources approved for the next ingestion milestones. It uses deliberately small, read-only samples so the source contracts can be understood before production ingestion code is designed.

| Status | Source | Dataset | What one record represents |
| --- | --- | --- | --- |
| Implemented | Alpaca | Asset metadata | The current reference-data snapshot for one stock |
| Implemented | Alpaca | Daily bars | One stock's aggregated trading activity for one day |
| Implemented | Alpaca | Trades | One executed transaction observed on the selected feed |
| Implemented | Alpaca | Quotes | One bid-and-ask update observed on the selected feed |
| Implemented | SEC EDGAR | Filing metadata | One regulatory filing submitted by one company |
| Planned | Alpaca | Minute bars | One stock's aggregated trading activity for one minute |
| Planned | Alpaca | Corporate actions | One dividend, split, merger, or other issuer action |
| Planned | Alpaca | Historical news | One article that can relate to several securities |
| Planned | SEC EDGAR | XBRL company facts | One reported accounting fact for a company and period |
| Planned | FINRA | Reg SHO daily volume | One symbol, facility, market, and trade-date observation |
| Planned | FRED | Macroeconomic observations | One value for one economic series and observation date |
| Planned | Nasdaq Trader | Trading halts | One halt or pause and its possible resumption information |

Google Cloud Storage and BigQuery are destinations, not source systems. A planned sample appearing here does not mean its ingestion, orchestration, or warehouse model has been implemented.

## Before running

1. Open the repository root in VS Code.
2. Run `uv sync --project ingestion --frozen`.
3. Select the Python kernel from `ingestion/.venv`.
4. Ensure the repository-root `.env` contains Alpaca credentials and `SEC_USER_AGENT`. Add `FRED_API_KEY` to run the optional FRED sample.

The notebook never prints credentials. The SEC user agent should identify the caller, for example `Your Name your.email@example.com`.

In [1]:
from datetime import UTC, date, datetime, timedelta
from xml.etree import ElementTree

import pandas as pd
import requests
from alpaca.data.enums import Adjustment, DataFeed
from alpaca.data.historical import NewsClient
from alpaca.data.historical.corporate_actions import CorporateActionsClient
from alpaca.data.requests import (
    CorporateActionsRequest,
    NewsRequest,
    StockBarsRequest,
    StockQuotesRequest,
    StockTradesRequest,
)
from alpaca.data.timeframe import TimeFrame
from equity_data_platform.config import get_settings
from equity_data_platform.sources.alpaca.assets import PARQUET_SCHEMA as ASSET_SCHEMA
from equity_data_platform.sources.alpaca.clients import (
    create_historical_client,
    create_trading_client,
)
from equity_data_platform.sources.alpaca.daily_bars import (
    PARQUET_SCHEMA as DAILY_BAR_SCHEMA,
)
from equity_data_platform.sources.alpaca.daily_bars import (
    DailyBarsBatchRequest,
    fetch_daily_bars,
)
from equity_data_platform.sources.alpaca.market_events import (
    QUOTE_SCHEMA,
    TRADE_SCHEMA,
)
from equity_data_platform.sources.sec.filings import (
    PARQUET_SCHEMA as SEC_FILING_SCHEMA,
)
from equity_data_platform.sources.sec.filings import (
    SecFilingsBatchRequest,
    fetch_sec_filings,
)
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

## Inspection parameters

The trades and quotes examples request only one minute. A full trading day can contain hundreds of thousands of quote updates, so full-day extraction belongs in the pipeline rather than an exploratory notebook.

In [2]:
settings = get_settings()

SAMPLE_SYMBOL = settings.symbols[0]
SAMPLE_TRADING_DATE = date(2026, 10, 27)
MARKET_WINDOW_START = datetime(2026, 10, 27, 13, 30, tzinfo=UTC)
MARKET_WINDOW_END = MARKET_WINDOW_START + timedelta(minutes=1)
SEC_START_DATE = date(2026, 10, 1)
SEC_END_DATE = date(2026, 10, 31)
CORPORATE_ACTION_START_DATE = date(2026, 1, 1)
CORPORATE_ACTION_END_DATE = date(2026, 11, 8)
NEWS_WINDOW_START = datetime(2026, 10, 27, tzinfo=UTC)
NEWS_WINDOW_END = NEWS_WINDOW_START + timedelta(days=1)

SEC_USER_AGENT_CONFIGURED = bool(settings.sec_user_agent)
FRED_API_KEY_CONFIGURED = bool(
    settings.fred_api_key and settings.fred_api_key.get_secret_value()
)

print("Configured symbols:", settings.symbols)
print("Sample symbol:", SAMPLE_SYMBOL)
print("Alpaca feed:", settings.feed)
print("SEC user agent configured:", SEC_USER_AGENT_CONFIGURED)
print("FRED API key configured:", FRED_API_KEY_CONFIGURED)
print("Intraday sample window:", MARKET_WINDOW_START, "to", MARKET_WINDOW_END)

Configured symbols: ('AAPL', 'MSFT', 'NVDA', 'AMZN', 'META')
Sample symbol: AAPL
Alpaca feed: iex
SEC user agent configured: True
FRED API key configured: True
Intraday sample window: 2026-10-27 13:30:00+00:00 to 2026-10-27 13:31:00+00:00


In [3]:
def inspect_frame(frame: pd.DataFrame, sample_rows: int = 5) -> None:
    print("Shape:", frame.shape)
    print("Columns:", frame.columns.tolist())
    display(frame.head(sample_rows))


def schema_frame(schema) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "field": [field.name for field in schema],
            "stored_type": [str(field.type) for field in schema],
            "nullable": [field.nullable for field in schema],
        }
    )

## 1. Alpaca asset metadata

Asset metadata describes what the security is and whether Alpaca currently allows particular trading operations. It is reference data, not a market event. The stable Alpaca asset ID is more durable than the ticker because a ticker can change.

In [4]:
trading_client = create_trading_client(settings, raw_data=True)
asset = trading_client.get_asset(SAMPLE_SYMBOL)

asset_frame = pd.DataFrame([asset])
inspect_frame(asset_frame)
display(asset_frame.T.rename(columns={0: "value"}))

Shape: (1, 16)
Columns: ['id', 'class', 'exchange', 'symbol', 'name', 'status', 'tradable', 'marginable', 'maintenance_margin_requirement', 'margin_requirement_long', 'margin_requirement_short', 'shortable', 'easy_to_borrow', 'borrow_status', 'fractionable', 'attributes']


,id,class,exchange,symbol,name,status,tradable,marginable,maintenance_margin_requirement,margin_requirement_long,margin_requirement_short,shortable,easy_to_borrow,borrow_status,fractionable,attributes
0,b0b6dd9d-8b9b-48a9-ba46-b9d54906e415,us_equity,NASDAQ,AAPL,Apple Inc. Common Stock,active,True,True,30,30,30,True,True,easy_to_borrow,True,"[fractional_eh_enabled, has_options, overnight_tradable]"


,value
id,b0b6dd9d-8b9b-48a9-ba46-b9d54906e415
class,us_equity
exchange,NASDAQ
symbol,AAPL
name,Apple Inc. Common Stock
status,active
tradable,True
marginable,True
maintenance_margin_requirement,30
margin_requirement_long,30


### Asset fields retained by the pipeline

The stored schema keeps provider fields and adds lineage such as `run_id`, `batch_key`, `asset_key`, `payload_fingerprint`, and `ingested_at`.

In [5]:
display(schema_frame(ASSET_SCHEMA))

,field,stored_type,nullable
0,provider_asset_id,string,False
1,symbol,string,False
2,name,string,True
3,asset_class,string,False
4,exchange,string,False
5,status,string,False
6,tradable,bool,False
7,marginable,bool,False
8,shortable,bool,False
9,fractionable,bool,False


## 2. Alpaca daily bars

One daily bar summarizes a stock's activity for one trading day. `open`, `high`, `low`, and `close` describe price movement. `volume` counts traded shares, `trade_count` counts transactions, and `vwap` is the volume-weighted average price.

In [6]:
historical_client = create_historical_client(settings)
daily_bar_request = DailyBarsBatchRequest(
    start_date=SAMPLE_TRADING_DATE,
    end_date=SAMPLE_TRADING_DATE,
    symbols=(SAMPLE_SYMBOL,),
    feed=settings.feed,
)
daily_bars = fetch_daily_bars(historical_client, daily_bar_request)
inspect_frame(daily_bars)

Shape: (1, 9)
Columns: ['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap']


,symbol,timestamp,open,high,low,close,volume,trade_count,vwap
0,AAPL,2026-10-27 04:00:00+00:00,343.165,347.77,342.41,347.1,1079424.0,23913.0,346.136337


In [7]:
display(schema_frame(DAILY_BAR_SCHEMA))

,field,stored_type,nullable
0,symbol,string,False
1,timestamp,"timestamp[us, tz=UTC]",False
2,open,"decimal128(20, 9)",False
3,high,"decimal128(20, 9)",False
4,low,"decimal128(20, 9)",False
5,close,"decimal128(20, 9)",False
6,volume,int64,False
7,trade_count,int64,False
8,vwap,"decimal128(20, 9)",False
9,source,string,False


## 3. Alpaca trades

A trade is an executed transaction. It reports where, when, at what price, and in what size a transaction occurred. The provider trade ID is used as part of the stable event identity. Conditions and tape codes carry market-structure context.

In [8]:
trade_request = StockTradesRequest(
    symbol_or_symbols=[SAMPLE_SYMBOL],
    start=MARKET_WINDOW_START,
    end=MARKET_WINDOW_END,
    feed=DataFeed.IEX,
)
trade_response = historical_client.get_stock_trades(trade_request)
trades = trade_response.df.reset_index()
inspect_frame(trades)

Shape: (259, 8)
Columns: ['symbol', 'timestamp', 'exchange', 'price', 'size', 'id', 'conditions', 'tape']


,symbol,timestamp,exchange,price,size,id,conditions,tape
0,AAPL,2026-10-27 13:30:00.077986+00:00,V,343.165,5.0,4,"[@, I]",C
1,AAPL,2026-10-27 13:30:00.101928+00:00,V,343.165,300.0,5,[@],C
2,AAPL,2026-10-27 13:30:00.114150+00:00,V,343.165,1.0,6,"[@, I]",C
3,AAPL,2026-10-27 13:30:00.135828+00:00,V,343.165,1.0,7,"[@, I]",C
4,AAPL,2026-10-27 13:30:00.166045+00:00,V,343.260,13.0,8,"[@, I]",C


In [9]:
display(schema_frame(TRADE_SCHEMA))

,field,stored_type,nullable
0,symbol,string,False
1,timestamp,"timestamp[us, tz=UTC]",False
2,provider_trade_id,int64,False
3,exchange,string,True
4,price,"decimal128(20, 9)",False
5,size,"decimal128(20, 9)",False
6,conditions,list<item: string>,True
7,tape,string,True
8,source,string,False
9,dataset,string,False


## 4. Alpaca quotes

A quote is the best displayed bid and ask update observed on the selected feed. The bid is the highest displayed buying price and the ask is the lowest displayed selling price. Their difference is the spread. Quote size describes displayed liquidity, not a completed transaction.

In [10]:
quote_request = StockQuotesRequest(
    symbol_or_symbols=[SAMPLE_SYMBOL],
    start=MARKET_WINDOW_START,
    end=MARKET_WINDOW_END,
    feed=DataFeed.IEX,
)
quote_response = historical_client.get_stock_quotes(quote_request)
quotes = quote_response.df.reset_index()
quotes["spread"] = quotes["ask_price"] - quotes["bid_price"]
quotes["midpoint"] = (quotes["ask_price"] + quotes["bid_price"]) / 2
inspect_frame(quotes)

Shape: (3479, 12)
Columns: ['symbol', 'timestamp', 'bid_price', 'bid_size', 'bid_exchange', 'ask_price', 'ask_size', 'ask_exchange', 'conditions', 'tape', 'spread', 'midpoint']


,symbol,timestamp,bid_price,bid_size,bid_exchange,ask_price,ask_size,ask_exchange,conditions,tape,spread,midpoint
0,AAPL,2026-10-27 13:30:00.001387+00:00,325.91,40.0,V,360.46,40.0,V,[R],C,34.55,343.185
1,AAPL,2026-10-27 13:30:00.068781+00:00,340.00,520.0,V,360.46,40.0,V,[R],C,20.46,350.230
2,AAPL,2026-10-27 13:30:00.172996+00:00,340.00,520.0,V,348.46,520.0,V,[R],C,8.46,344.230
3,AAPL,2026-10-27 13:30:00.329013+00:00,343.04,40.0,V,348.46,520.0,V,[R],C,5.42,345.750
4,AAPL,2026-10-27 13:30:00.329041+00:00,343.04,40.0,V,343.29,40.0,V,[R],C,0.25,343.165


In [11]:
display(schema_frame(QUOTE_SCHEMA))

,field,stored_type,nullable
0,symbol,string,False
1,timestamp,"timestamp[us, tz=UTC]",False
2,bid_price,"decimal128(20, 9)",False
3,bid_size,"decimal128(20, 9)",False
4,bid_exchange,string,True
5,ask_price,"decimal128(20, 9)",False
6,ask_size,"decimal128(20, 9)",False
7,ask_exchange,string,True
8,quote_size_unit,string,False
9,conditions,list<item: string>,True


## 5. SEC EDGAR filing metadata

SEC EDGAR is independent from Alpaca. A filing record describes a document submitted to the regulator, such as a 10-K annual report, 10-Q quarterly report, or 8-K current report. `acceptance_datetime` is especially important because it indicates when the filing became publicly available.

This first SEC pipeline stores filing metadata. It does not yet ingest the financial facts inside each filing.

In [12]:
if not settings.sec_user_agent:
    raise ValueError("Add SEC_USER_AGENT to the repository-root .env before calling SEC EDGAR")

sec_request = SecFilingsBatchRequest(
    start_date=SEC_START_DATE,
    end_date=SEC_END_DATE,
    symbols=(SAMPLE_SYMBOL,),
)
sec_session = requests.Session()
filing_records = fetch_sec_filings(
    sec_request,
    user_agent=settings.sec_user_agent,
    session=sec_session,
)
filings = pd.DataFrame(filing_records)
inspect_frame(filings)

Shape: (15, 18)
Columns: ['cik', 'symbol', 'entity_name', 'exchange', 'accession_number', 'filing_date', 'report_date', 'acceptance_datetime', 'form', 'act', 'file_number', 'film_number', 'items', 'document_size', 'is_xbrl', 'is_inline_xbrl', 'primary_document', 'primary_document_description']


,cik,symbol,entity_name,exchange,accession_number,filing_date,report_date,acceptance_datetime,form,act,file_number,film_number,items,document_size,is_xbrl,is_inline_xbrl,primary_document,primary_document_description
0,0000320193,AAPL,Apple Inc.,Nasdaq,0001140361-26-041201,2026-10-29,2026-10-27,2026-10-29T22:30:14.000Z,4,,,,,4822,0,0,xslF345X06/form4.xml,FORM 4
1,0000320193,AAPL,Apple Inc.,Nasdaq,0001140361-26-040540,2026-10-22,2026-10-20,2026-10-22T22:32:20.000Z,4,,,,,4822,0,0,xslF345X06/form4.xml,FORM 4
2,0000320193,AAPL,Apple Inc.,Nasdaq,0001140361-26-040127,2026-10-19,2026-10-15,2026-10-19T22:32:55.000Z,4,,,,,24526,0,0,xslF345X06/form4.xml,FORM 4
3,0000320193,AAPL,Apple Inc.,Nasdaq,0001140361-26-040126,2026-10-19,2026-10-15,2026-10-19T22:30:18.000Z,4,,,,,20074,0,0,xslF345X06/form4.xml,FORM 4
4,0000320193,AAPL,Apple Inc.,Nasdaq,0001950047-26-010286,2026-10-16,,2026-10-16T20:29:07.000Z,144,33,001-36743,261466426,,4810,0,0,xsl144X01/primary_doc.xml,


In [13]:
display(schema_frame(SEC_FILING_SCHEMA))

,field,stored_type,nullable
0,cik,string,False
1,symbol,string,False
2,entity_name,string,False
3,exchange,string,True
4,accession_number,string,False
5,filing_date,date32[day],False
6,report_date,date32[day],True
7,acceptance_datetime,"timestamp[us, tz=UTC]",True
8,form,string,False
9,act,string,True


# Planned ingestion sources

The following sections inspect approved source candidates before their ingestion contracts are implemented. They display provider fields only. No Parquet schema is shown yet because selecting and typing the retained fields is part of the ingestion work.

## 6. Alpaca minute bars — planned

Minute bars have the same OHLCV structure as daily bars, but their grain is one security and one minute. They provide an efficient timeline for event-window analysis without repeatedly aggregating every trade.

In [14]:
minute_bar_request = StockBarsRequest(
    symbol_or_symbols=[SAMPLE_SYMBOL],
    timeframe=TimeFrame.Minute,
    start=MARKET_WINDOW_START,
    end=MARKET_WINDOW_START + timedelta(minutes=5),
    adjustment=Adjustment.RAW,
    feed=DataFeed.IEX,
)
minute_bars = historical_client.get_stock_bars(minute_bar_request).df.reset_index()
inspect_frame(minute_bars)

Shape: (6, 9)
Columns: ['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap']


,symbol,timestamp,open,high,low,close,volume,trade_count,vwap
0,AAPL,2026-10-27 13:30:00+00:00,343.165,344.430,343.165,343.550,11917.0,259.0,343.730939
1,AAPL,2026-10-27 13:31:00+00:00,343.530,344.940,343.310,344.340,7001.0,121.0,344.133884
2,AAPL,2026-10-27 13:32:00+00:00,344.450,344.875,344.030,344.595,4553.0,90.0,344.420293
3,AAPL,2026-10-27 13:33:00+00:00,344.540,345.090,343.850,344.020,4608.0,97.0,344.533305
4,AAPL,2026-10-27 13:34:00+00:00,344.040,344.225,343.790,343.790,4657.0,94.0,344.029825


## 7. Alpaca corporate actions — planned

Corporate actions explain structural changes such as cash dividends, stock dividends, forward and reverse splits, mergers, spin-offs, and name changes. Different action types have different fields, so the source returns records grouped by action type.

In [15]:
corporate_actions_client = CorporateActionsClient(
    api_key=settings.alpaca_api_key.get_secret_value(),
    secret_key=settings.alpaca_secret_key.get_secret_value(),
    raw_data=True,
)
corporate_actions_response = corporate_actions_client.get_corporate_actions(
    CorporateActionsRequest(
        symbols=[SAMPLE_SYMBOL],
        start=CORPORATE_ACTION_START_DATE,
        end=CORPORATE_ACTION_END_DATE,
        limit=100,
    )
)
corporate_action_records = [
    {"corporate_action_type": action_type, **record}
    for action_type, records in corporate_actions_response.items()
    for record in records
]
corporate_actions = pd.DataFrame(corporate_action_records)
inspect_frame(corporate_actions)

Shape: (3, 11)
Columns: ['corporate_action_type', 'cusip', 'ex_date', 'foreign', 'id', 'payable_date', 'process_date', 'rate', 'record_date', 'special', 'symbol']


,corporate_action_type,cusip,ex_date,foreign,id,payable_date,process_date,rate,record_date,special,symbol
0,cash_dividends,037833100,2026-02-09,False,5f891a9b-393d-47ed-b449-b2306f13c63b,2026-02-12,2026-02-12,0.26,2026-02-09,False,AAPL
1,cash_dividends,037833100,2026-05-11,False,a2df24db-32ca-430e-98a9-f8c7992e3468,2026-05-14,2026-05-14,0.27,2026-05-11,False,AAPL
2,cash_dividends,037833100,2026-08-10,False,b1373f73-2e44-4947-931a-ffa630b4ea37,2026-08-13,2026-08-13,0.27,2026-08-10,False,AAPL


## 8. Alpaca historical news — planned

A news article contains publication times, headline, summary, source, URL, images, and a list of related symbols. Because one article can mention several securities, the warehouse will need an article table plus an article-to-security bridge rather than one duplicated article per symbol.

In [16]:
news_client = NewsClient(
    api_key=settings.alpaca_api_key.get_secret_value(),
    secret_key=settings.alpaca_secret_key.get_secret_value(),
    raw_data=True,
)
news_response = news_client.get_news(
    NewsRequest(
        symbols=SAMPLE_SYMBOL,
        start=NEWS_WINDOW_START,
        end=NEWS_WINDOW_END,
        limit=5,
        include_content=False,
    )
)
news = pd.DataFrame(news_response.get("news", []))
inspect_frame(news)

Shape: (5, 11)
Columns: ['author', 'content', 'created_at', 'headline', 'id', 'images', 'source', 'summary', 'symbols', 'updated_at', 'url']


,author,content,created_at,headline,id,images,source,summary,symbols,updated_at,url
0,The Arora Report,,2026-10-27T18:42:47Z,"AI's Biggest Constraint Could Be Capital Availability—Anthropic, Meta And SpaceX Put Markets To The Test",62632804,"[{'size': 'large', 'url': 'https://cdn.benzinga.com/files/images/story/2026/10/27/Anthropic-Logo.jpg?width=2048&heig...",benzinga,\n\n\n\nAI Financing Deluge\n\n\n\nPlease click here for an enlarged chart of Direxion Daily Semiconductor Bull 3X E...,"[AAPL, AMZN, GOOG, META, MSFT, NVDA, QQQ, SOXL, SPY, TSLA]",2026-10-27T19:55:10Z,https://www.benzinga.com/opinion/26/10/62632804/ai-biggest-constraint-could-be-capital-availability-anthropic-meta-a...
1,,,2026-10-27T18:01:26Z,Apple Creator Studio Update Adds MotionVFX and AI Assistants,105163186,[],fiscal.ai,"Apple announced a major update to its Creator Studio software suite, integrating MotionVFX tools and new AI-powered ...",[AAPL],2026-10-27T18:01:26Z,https://appleosophy.com/wire/x2n4dvt/
2,Benzinga Newsdesk,,2026-10-27T17:01:19Z,"CNBC Halftime Report Final Trades: Apple, UnitedHealth Group, Delta Air Lines, American Tower",62629237,[],benzinga,,"[AAPL, AMT, DAL, UNH]",2026-10-27T17:01:19Z,https://www.benzinga.com/quote/AAPL
3,Anusuya Lahiri,,2026-10-27T16:57:07Z,What Is Going On With Apple Stock on Tuesday?,62629051,"[{'size': 'large', 'url': 'https://cdn.benzinga.com/files/images/story/2026/10/27/Apple-iPhone-Duo.jpg?width=2048&he...",benzinga,"Apple stock holds steady as iPhone 18 Pro sales climb 3.1% globally, driven by a 29% spike in China, ahead of the co...","[AAPL, FNGU, SPMO, USMC]",2026-10-27T16:57:07Z,https://www.benzinga.com/markets/tech/26/10/62629051/what-is-going-on-with-apple-stock-on-tuesday
4,Benzinga Insights,,2026-10-27T15:00:23Z,Growth Leads Sectors In Tuesday Trading As Defensives Trail,62623952,"[{'size': 'large', 'url': 'https://cdn.benzinga.com/files/images/story/2026/10/27/Charts-Of-Financial-Instruments-Wi...",benzinga,"Seven sectors are higher and four are lower in Tuesday&#39;s regular session, with growth and cyclical sectors split...","[AAPL, AMT, AMZN, APD, AVGO, COP, COST, CVX, DIA, ECL, EOG, HD, IWM, NEM, ORCL, PEP, PLD, QQQ, SPG, SPY, TSLA, WMT, ...",2026-10-27T15:00:23Z,https://www.benzinga.com/etfs/sector-etfs/26/10/62623952/growth-leads-sectors-in-tuesday-trading-as-defensives-trail


## 9. SEC EDGAR XBRL company facts — planned

The Company Facts API returns accounting concepts extracted from filings. A concept can contain many observations with different units, fiscal periods, forms, filing dates, and accession numbers. The eventual raw model should therefore remain long-form and preserve the SEC taxonomy and concept name.

In [17]:
sample_cik = str(filings.iloc[0]["cik"]) if not filings.empty else "0000320193"
company_facts_response = sec_session.get(
    f"https://data.sec.gov/api/xbrl/companyfacts/CIK{sample_cik}.json",
    headers={"User-Agent": settings.sec_user_agent},
    timeout=30,
)
company_facts_response.raise_for_status()
company_facts_payload = company_facts_response.json()

concepts = pd.DataFrame(
    [
        {
            "taxonomy": taxonomy,
            "concept": concept,
            "label": details.get("label"),
            "description": details.get("description"),
            "units": list(details.get("units", {})),
        }
        for taxonomy, taxonomy_facts in company_facts_payload.get("facts", {}).items()
        for concept, details in taxonomy_facts.items()
    ]
)
print("Entity:", company_facts_payload.get("entityName"))
inspect_frame(concepts, sample_rows=10)

Entity: Apple Inc.
Shape: (520, 5)
Columns: ['taxonomy', 'concept', 'label', 'description', 'units']


,taxonomy,concept,label,description,units
0,dei,EntityCommonStockSharesOutstanding,"Entity Common Stock, Shares Outstanding",Indicate number of shares or other units outstanding of each of registrant's classes of capital or common stock or o...,[shares]
1,dei,EntityPublicFloat,Entity Public Float,The aggregate market value of the voting and non-voting common equity held by non-affiliates computed by reference t...,[USD]
2,us-gaap,AccountsPayable,Accounts Payable (Deprecated 2009-01-31),Carrying value as of the balance sheet date of liabilities incurred (and for which invoices have typically been rece...,[USD]
3,us-gaap,AccountsPayableCurrent,"Accounts Payable, Current",Carrying value as of the balance sheet date of liabilities incurred (and for which invoices have typically been rece...,[USD]
4,us-gaap,AccountsReceivableNetCurrent,"Accounts Receivable, after Allowance for Credit Loss, Current","Amount, after allowance for credit loss, of right to consideration from customer for product sold and service render...",[USD]
5,us-gaap,AccruedIncomeTaxesCurrent,"Accrued Income Taxes, Current",Carrying amount as of the balance sheet date of the unpaid sum of the known and estimated amounts payable to satisfy...,[USD]
6,us-gaap,AccruedIncomeTaxesNoncurrent,"Accrued Income Taxes, Noncurrent",Carrying amount as of the balance sheet date of the unpaid sum of the known and estimated amounts payable to satisfy...,[USD]
7,us-gaap,AccruedLiabilities,Accrued Liabilities (Deprecated 2009-01-31),"Carrying value as of the balance sheet date of obligations incurred and payable, pertaining to costs that are statut...",[USD]
8,us-gaap,AccruedLiabilitiesCurrent,"Accrued Liabilities, Current","Carrying value as of the balance sheet date of obligations incurred and payable, pertaining to costs that are statut...",[USD]
9,us-gaap,AccruedMarketingCostsCurrent,"Accrued Marketing Costs, Current","Carrying value as of the balance sheet date of obligations incurred through that date and payable for the marketing,...",[USD]


In [18]:
preferred_concepts = [
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "NetIncomeLoss",
    "Assets",
]
available_us_gaap = company_facts_payload.get("facts", {}).get("us-gaap", {})
sample_concept_name = next(
    (name for name in preferred_concepts if name in available_us_gaap),
    next(iter(available_us_gaap)),
)
sample_concept = available_us_gaap[sample_concept_name]
sample_unit, sample_observations = next(iter(sample_concept["units"].items()))
print("Concept:", sample_concept_name)
print("Unit:", sample_unit)
inspect_frame(pd.DataFrame(sample_observations).sort_values("filed", ascending=False))

Concept: RevenueFromContractWithCustomerExcludingAssessedTax
Unit: USD
Shape: (120, 9)
Columns: ['start', 'end', 'val', 'accn', 'fy', 'fp', 'form', 'filed', 'frame']


,start,end,val,accn,fy,fp,form,filed,frame
101,2023-10-01,2024-09-28,391035000000,0000320193-26-000026,2026,FY,10-K,2026-11-03,CY2024
113,2024-09-29,2025-09-27,416161000000,0000320193-26-000026,2026,FY,10-K,2026-11-03,CY2025
119,2025-09-28,2026-09-26,478834000000,0000320193-26-000026,2026,FY,10-K,2026-11-03,CY2026
109,2024-09-29,2025-06-28,313695000000,0000320193-26-000020,2026,Q3,10-Q,2026-07-31,NaN
111,2025-03-30,2025-06-28,94036000000,0000320193-26-000020,2026,Q3,10-Q,2026-07-31,CY2025Q2


## 10. FINRA Reg SHO daily short-sale volume — planned

FINRA reports daily short-sale volume by symbol, market, and reporting facility. This is executed short-sale volume reported to FINRA facilities. It is not short interest and must not be labelled as such. The public dataset currently provides a rolling twelve-month window.

In [19]:
finra_url = "https://api.finra.org/data/group/otcMarket/name/regShoDaily"
finra_query = {
    "compareFilters": [
        {
            "compareType": "EQUAL",
            "fieldName": "securitiesInformationProcessorSymbolIdentifier",
            "fieldValue": SAMPLE_SYMBOL,
        }
    ],
    "limit": 10,
}
finra_response = requests.post(
    finra_url,
    json=finra_query,
    headers={"Accept": "application/json"},
    timeout=30,
)
finra_response.raise_for_status()
finra_short_volume = pd.DataFrame(finra_response.json())
inspect_frame(finra_short_volume)

Shape: (10, 7)
Columns: ['reportingFacilityCode', 'totalParQuantity', 'shortParQuantity', 'marketCode', 'tradeReportDate', 'securitiesInformationProcessorSymbolIdentifier', 'shortExemptParQuantity']


,reportingFacilityCode,totalParQuantity,shortParQuantity,marketCode,tradeReportDate,securitiesInformationProcessorSymbolIdentifier,shortExemptParQuantity
0,NCTRF,154130,69397,B,2026-01-06,AAPL,20
1,NQTRF,17699980,4891733,Q,2026-01-06,AAPL,85087
2,NYTRF,588753,198716,N,2026-01-06,AAPL,250
3,NCTRF,120279,62763,B,2026-01-05,AAPL,0
4,NQTRF,15035913,4898542,Q,2026-01-05,AAPL,41271


## 11. FRED macroeconomic observations — planned

FRED organizes economic data into series. One observation contains a series value for an observation date plus real-time dates that identify the revision window. The production design should preserve series metadata and vintages because macroeconomic values can be revised. This sample requires a free FRED API key.

In [20]:
if not FRED_API_KEY_CONFIGURED:
    print("Skipped: add FRED_API_KEY to .env to inspect FRED observations.")
    fred_observations = pd.DataFrame()
else:
    fred_response = requests.get(
        "https://api.stlouisfed.org/fred/series/observations",
        params={
            "series_id": "DGS10",
            "api_key": settings.fred_api_key.get_secret_value(),
            "file_type": "json",
            "observation_start": "2026-10-01",
            "observation_end": "2026-10-31",
        },
        timeout=30,
    )
    fred_response.raise_for_status()
    fred_observations = pd.DataFrame(fred_response.json()["observations"])
    inspect_frame(fred_observations)

Shape: (22, 4)
Columns: ['realtime_start', 'realtime_end', 'date', 'value']


,realtime_start,realtime_end,date,value
0,2026-11-06,2026-11-06,2026-10-01,5.24
1,2026-11-06,2026-11-06,2026-10-02,5.28
2,2026-11-06,2026-11-06,2026-10-05,5.31
3,2026-11-06,2026-11-06,2026-10-06,5.27
4,2026-11-06,2026-11-06,2026-10-07,5.28


## 12. Nasdaq Trader trading halts — planned

The free Nasdaq Trader RSS feed reports current halt and pause information for Nasdaq-listed and other exchange-listed securities. It includes the halt time and reason plus resumption times when available. The feed should not be polled more often than once per minute.

In [21]:
nasdaq_response = requests.get(
    "https://www.nasdaqtrader.com/rss.aspx?feed=tradehalts",
    headers={"User-Agent": settings.sec_user_agent},
    timeout=30,
)
nasdaq_response.raise_for_status()
nasdaq_root = ElementTree.fromstring(nasdaq_response.content)
nasdaq_namespace = "{http://www.nasdaqtrader.com/}"
nasdaq_halts = pd.DataFrame(
    [
        {
            child.tag.removeprefix(nasdaq_namespace): child.text
            for child in item
            if child.tag != "description"
        }
        for item in nasdaq_root.findall(".//item")
    ]
)
inspect_frame(nasdaq_halts)

Shape: (23, 12)
Columns: ['title', 'pubDate', 'HaltDate', 'HaltTime', 'IssueSymbol', 'IssueName', 'Market', 'ReasonCode', 'PauseThresholdPrice', 'ResumptionDate', 'ResumptionQuoteTime', 'ResumptionTradeTime']


,title,pubDate,HaltDate,HaltTime,IssueSymbol,IssueName,Market,ReasonCode,PauseThresholdPrice,ResumptionDate,ResumptionQuoteTime,ResumptionTradeTime
0,STKH,"Fri, 06 Nov 2026 05:00:00 GMT",11/06/2026,19:50:00.000,STKH,Steakholder Foods Ltd. ADS,NASDAQ,T1,None,None,None,None
1,WCT,"Fri, 06 Nov 2026 05:00:00 GMT",11/06/2026,19:50:00.000,WCT,Wellchange Holdings Cl A Ord,NASDAQ,T1,None,None,None,None
2,ALPS,"Fri, 06 Nov 2026 05:00:00 GMT",11/06/2026,19:50:00.000,ALPS,ALPS Group Inc Ordinary Share,NASDAQ,T1,None,None,None,None
3,NXTS,"Fri, 06 Nov 2026 05:00:00 GMT",11/06/2026,19:50:00.000,NXTS,Nexentis Tech Inc Cmn,NASDAQ,T1,None,None,None,None
4,SMXWW,"Fri, 06 Nov 2026 05:00:00 GMT",11/06/2026,19:50:00.000,SMXWW,SMX Sec Mtrs PLC Warrants,NASDAQ,T1,None,None,None,None


## How the datasets relate

The tables should remain separate because they describe different business processes and grains. They can still be explored together through shared symbol/security keys and time windows.

- Asset metadata tells us which security the symbol represents.
- Daily and minute bars summarize activity at different time grains.
- Trades show the transactions inside a period.
- Quotes show the changing bid, ask, spread, and displayed size inside that period.
- Filings, news, corporate actions, and halts provide event context.
- SEC company facts provide fundamental company context.
- FINRA provides additional daily trading-activity context.
- FRED provides macroeconomic context.

A later event-explorer mart can select a stock and time interval, then return its bar plus aggregated trades, quote statistics, and company events. Raw trades and quotes should not be joined directly to one another because that would create a many-to-many row explosion.

In [22]:
inventory = pd.DataFrame(
    [
        {"status": "implemented", "source": "Alpaca", "dataset": "assets", "sample_rows": len(asset_frame)},
        {"status": "implemented", "source": "Alpaca", "dataset": "daily_bars", "sample_rows": len(daily_bars)},
        {"status": "implemented", "source": "Alpaca", "dataset": "trades", "sample_rows": len(trades)},
        {"status": "implemented", "source": "Alpaca", "dataset": "quotes", "sample_rows": len(quotes)},
        {"status": "implemented", "source": "SEC EDGAR", "dataset": "filings", "sample_rows": len(filings)},
        {"status": "planned", "source": "Alpaca", "dataset": "minute_bars", "sample_rows": len(minute_bars)},
        {"status": "planned", "source": "Alpaca", "dataset": "corporate_actions", "sample_rows": len(corporate_actions)},
        {"status": "planned", "source": "Alpaca", "dataset": "news", "sample_rows": len(news)},
        {"status": "planned", "source": "SEC EDGAR", "dataset": "xbrl_concepts", "sample_rows": len(concepts)},
        {"status": "planned", "source": "FINRA", "dataset": "reg_sho_daily", "sample_rows": len(finra_short_volume)},
        {"status": "planned", "source": "FRED", "dataset": "macro_observations", "sample_rows": len(fred_observations)},
        {"status": "planned", "source": "Nasdaq Trader", "dataset": "trading_halts", "sample_rows": len(nasdaq_halts)},
    ]
)
display(inventory)

,status,source,dataset,sample_rows
0,implemented,Alpaca,assets,1
1,implemented,Alpaca,daily_bars,1
2,implemented,Alpaca,trades,259
3,implemented,Alpaca,quotes,3479
4,implemented,SEC EDGAR,filings,15
5,planned,Alpaca,minute_bars,6
6,planned,Alpaca,corporate_actions,3
7,planned,Alpaca,news,5
8,planned,SEC EDGAR,xbrl_concepts,520
9,planned,FINRA,reg_sho_daily,10
